# Phase 4 — the sealed split, and what a search costs

Every score this project has ever reported came off rows the search was allowed to look at.

The validation half is held out from *training*, which is not the same as held out from
*choosing*. Twenty tables get built, each one scored on those same 292 rows, and the best
one is kept. The winner is then the maximum of twenty draws — and the maximum of twenty
draws is bigger than the truth even when nothing improved at all.

We know the size of that effect only because two of our datasets happen to have a Kaggle
leaderboard attached:

| run | reported | truth | optimism |
|---|---|---|---|
| house-prices run 2 (17 experiments) | 0.12162 | 0.12368 | +0.00206 |
| house-prices run 3 (24 experiments) | 0.11354 | 0.12298 | **+0.00944** |
| seguro v2 (11 experiments) | 0.28350 | 0.26522 | **+0.01828** |

1.4x the experiments, 4.6x the optimism. On a dataset with no competition attached we had
no way to measure this at all.

This notebook builds the third split that measures it in-house: rows nothing in the run can
reach, scored exactly once, at the very end. `research/archive/decisions/0023` decided it;
`research/archive/decisions/0041` is the record of how it was built and the four calls 0023 left open.

As always: you drive it by hand first, and the LLM shows up only at the end.

## 1. A dataset where we know the answer

The trick that makes this notebook checkable: we generate **two** samples from the same
process. One becomes the run's input, and the other never touches the run at all.

That second sample is the ground truth — an unlimited supply of genuinely unseen rows. At
the end we can ask whether the sealed score, measured on 120 rows inside the run, lands
anywhere near what the model actually does on a fresh population.

In [ ]:
import os
import tempfile

import numpy as np
import pandas as pd

CITY_PREMIUM = {"berlin": 25_000, "hamburg": 8_000, "koeln": 0, "essen": -6_000}


def sample_houses(n, seed):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "area": rng.uniform(30, 200, n),
        "rooms": rng.integers(1, 7, n),
        "city": rng.choice(list(CITY_PREMIUM), n),
    })
    df["price"] = (
        df["area"] * 900
        + df["rooms"] * 4_000
        + df["city"].map(CITY_PREMIUM)
        + rng.normal(0, 6_000, n)
        + 40_000
    )
    return df


run_data = sample_houses(600, seed=0)     # everything the run will ever see
fresh = sample_houses(4_000, seed=999)    # the truth: never enters the run

workdir = tempfile.mkdtemp(prefix="sealed_demo_")
data_path = os.path.join(workdir, "houses.csv")
run_data.to_csv(data_path, index=False)
run_data.head()

## 2. Cutting three ways

`holdout_fraction` is a fraction of the **whole input**, exactly like `val_fraction`. Both
come off the top; the work half is what is left.

The obvious-looking alternative is to carve the sealed rows out of the existing validation
half, leaving the work half untouched — it sounds free. It is not: it halves the validation
half, so `val_se` grows by about 1.4x on the number that ranks *every table in the run*.
That is paying for honest reporting with noise in selection, and 0022 is the record of what
noisy selection costs.

Run both and compare.

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.harness.stop import StopPolicy
from tabular_agent.harness import journal, tools
from tabular_agent.io import load_table
from tabular_agent.taskspec import TaskSpec


def new_session(name, **kwargs):
    return RunSession(
        os.path.join(workdir, name), data_path, TaskSpec(target="price"), **kwargs
    )


plain = new_session("plain")
# `patience` is turned up only so section 4 can build a dozen junk tables in a
# row. The stop rule would refuse the seventh, and refusing it is the correct
# behaviour — see the note there.
sealed = new_session("sealed", holdout_fraction=0.2, stop_policy=StopPolicy(patience=30))

print("without a seal:", len(load_table(plain.data_path)), "work /",
      len(load_table(plain.val_path)), "val")
print("with a seal   :", len(load_table(sealed.data_path)), "work /",
      len(load_table(sealed.val_path)), "val /",
      len(load_table(sealed.holdout_path)), "sealed")
print()
print("holdout_path when off:", plain.holdout_path, " <- opt-in, and off by default")

### The validation rows did not move

This is the property worth checking by hand, because everything this project does is a
comparison against an earlier run.

Under a random split the validation rows are taken off the front of **one permutation**, so
which rows are validation depends only on `n` and the seed — not on whether a holdout was
asked for. Turning the seal on shrinks the work half and leaves the exam alone.

In [ ]:
pd.testing.assert_frame_equal(load_table(plain.val_path), load_table(sealed.val_path))
print("validation halves are identical, row for row")

# ... and no row is in two places, or in none.
days = pd.concat([
    load_table(p)["price"] for p in (sealed.data_path, sealed.val_path, sealed.holdout_path)
])
print("rows covered exactly once:", len(days) == 600 == days.nunique())

## 3. The seal

`resolve_table` is an allowlist: nothing outside `tables/` resolves, ever. The sealed file
sits in `data/` next to the validation half, so it is unreachable for exactly the same
reason — no new mechanism, no new thing to keep in sync.

Try to reach it the way an agent would stumble into it: by naming the file.

In [ ]:
for attempt in ["data/holdout.parquet", sealed.holdout_path, "../sealed/data/holdout.parquet"]:
    try:
        sealed.resolve_table(attempt)
        print("REACHED IT —", attempt)
    except ValueError as e:
        print(f"refused: {str(e)[:60]}...")

print()
print("what the agent can see:", sealed.available_tables(), "(nothing yet — no tables built)")

## 4. Manufacturing a winner's curse

To measure optimism you first have to produce some. That takes no bad faith at all — just a
search that tries things.

Below, twelve tables, each adding one deterministic-but-meaningless column. Not one of them
carries information about `price`. Every one of them gets scored by the probe on the same
120 validation rows, and we keep the best.

**The stop rule would not allow this.** At the default `patience=6` the seventh junk table
in a row is refused, because six tables with no real gain is the harness deciding the run
is done (0012, 0014). We raised it in section 2 purely to manufacture the effect. Keep that
in mind when reading the numbers below: this is what the stop rule is *for*, seen from the
other side.

(The columns have to be deterministic, not random: 0011's stability check re-runs the code and
compares, so a genuinely random column is refused. `sin(area * k)` is junk that survives
being run twice, which is the honest version of what an agent does when it tries a ratio
that turns out to mean nothing.)

In [ ]:
tools.inspect_data(sealed)
spec = sealed.resolved()
print("metric:", spec.metric, "| target_transform:", spec.target_transform)

JUNK = """
import numpy as np
def transform_features(df):
    df = df.copy()
    df["j{k}"] = np.sin(df["area"] * {k}.0) + np.cos(df["rooms"] * {k}.0)
    return df
"""

rows = []
for k in range(1, 13):
    out = tools.transform_features(sealed, JUNK.format(k=k))
    rows.append({"table": out["table"], "val": out.get("probe_score")})

junk = pd.DataFrame(rows).sort_values("val").reset_index(drop=True)
print(f"\n12 junk tables, all scored on the same {len(load_table(sealed.val_path))} val rows:")
print(junk.head(4).to_string(index=False))
print("   ...")
print(junk.tail(2).to_string(index=False))
print(f"\nbest {junk['val'].iloc[0]:.5f} / worst {junk['val'].iloc[-1]:.5f}"
      f"  -> spread {junk['val'].iloc[-1] - junk['val'].iloc[0]:.5f} of rmsle, from nothing at all")

Every one of those tables is the same information plus noise, and they are spread over
thousands of currency units. Nothing improved; the ranking is measuring which arrangement of
noise happens to suit 120 particular rows.

A search takes the top of that list. That is where optimism comes from — not from a mistake,
but from the maximum being a biased estimator of the truth.

## 5. Finalize breaks the seal, once

`finalize_run` picks the winner **first**, on the validation half, exactly as it always has.
Only then does it open the sealed rows — and it scores the whole terminal ballot, not just
the winner.

That is 0041's one substantive addition to 0023, and it is deliberately in tension with it:
four numbers on a sealed set, ranked, is a third leaderboard waiting to happen. Three things
hold the line, and none of them is advice:

- `_score_on_holdout` takes the winner **as an argument**. There is no ordering in which it
  could influence the choice.
- The scores are written under `journal["holdout"]`, never onto the leaderboard, and
  `set_holdout` is write-once — "scored exactly once" is auditable from the journal alone.
- Every entry carries `selected`, so the number the search steered towards is never confused
  with the number that judged it.

In [ ]:
final = tools.finalize_run(sealed)

print("winner     :", final["best_experiment_id"], final["model_type"])
print(f"val_score  : {final['val_score']:.5f}   <- what the search steered towards")
print(f"holdout    : {final['holdout_score']:.5f}   <- rows nothing in the run could read")
print(f"optimism   : {final['optimism']:+.5f}  <- positive means val flattered")

### The ballot

Same table, same sealed rows, four models. The pair that matters is the shipped one against
the ones it beat.

0039 and 0040 argued that optimism attaches to the **selected measurement** rather than to
the model or the table — and named their own confound: in all three pairs they had, the
selected model was lgbm and the unselected one elasticnet, so "selection inflates" and "lgbm
inflates more" predict the same thing. A winner and its runners-up on one sealed set is that
comparison without the confound, and it costs nothing, because the fits already exist.

In [ ]:
record = journal.load(sealed.run_dir)["holdout"]
ballot = pd.DataFrame(record["scores"])
print(f"sealed rows: {record['n_rows']}\n")
print(ballot[["model_type", "selected", "val_score", "holdout_score", "optimism"]]
      .round(5).to_string(index=False))

## 6. Is the sealed number actually honest?

The whole claim is that 120 rows nobody looked at behave like rows that do not exist yet. We
can check it, because we kept 4,000 rows from the same process that never entered the run.

Score the shipped bundle on them. Three numbers: what the run reported, what the seal said,
and what the model is really worth.

In [ ]:
from tabular_agent.metrics import compute_metric
from tabular_agent.pipeline import load_bundle, predict_bundle

bundle = load_bundle(os.path.join(sealed.run_dir, final["bundle_dir"]))
preds = predict_bundle(bundle, fresh.drop(columns=["price"]))
truth = compute_metric(spec.metric, fresh["price"], preds)

print(f"val_score  (search selected on these) : {final['val_score']:.5f}")
print(f"holdout    (sealed, scored once)      : {final['holdout_score']:.5f}")
print(f"truth      (4,000 fresh rows)         : {truth:.5f}")
print()
print(f"val    was off by {abs(final['val_score'] - truth):.5f}")
print(f"sealed was off by {abs(final['holdout_score'] - truth):.5f}")

Two honest caveats about that comparison, both of which 0023 names:

- **The sealed number is not precise, only unbiased.** 120 rows have their own standard
  error; the report prints it beside the score and flags the split as thin below 30 rows. A
  small gap is not a clean bill of health.
- **Run it with a different seed and the numbers move.** The point is the *direction* — the
  validation score is the one with a thumb on the scale, because it is the one the search
  was allowed to consult.

## 7. The report

The gap is printed, not just stored, and both numbers appear — reporting only the sealed
score would throw away the diagnostic, which is the thing no run has ever had.

In [ ]:
text = open(os.path.join(sealed.run_dir, final["report_path"])).read()
start = text.index("## What the search cost")
print(text[:text.index("\n\n>")])
print("\n...\n")
print(text[start:text.index("## Per-fold")])

## 8. Where the gap means less than it looks: ordered splits

On a time-ordered dataset the three slices are `work | val | holdout` **in time order**. The
sealed rows are the latest period, because the sealed set stands in for unseen data and
unseen data is always the future — the same argument 0029 makes for the validation half,
one slice further out.

Two consequences follow, and both are the price of that choice:

1. Turning the seal on **moves the validation rows earlier**, so an ordered run with a seal
   is not row-for-row comparable to one without. (Under a random split it was.)
2. val and holdout are now *different periods*, so the gap between them mixes selection
   optimism with **drift**. This run cannot separate them.

Build a series that drifts and watch the second one happen with no search involved at all.

In [ ]:
n = 900
t = np.arange(n)
rng = np.random.default_rng(3)
drifting = pd.DataFrame({
    "day": t,
    "x": rng.normal(size=n),
    # a level that climbs steadily: nothing to search, the world simply moves
    "y": 100.0 + 0.05 * t + rng.normal(scale=1.0, size=n),
})
drift_path = os.path.join(workdir, "drift.csv")
drifting.to_csv(drift_path, index=False)

ordered = RunSession(
    os.path.join(workdir, "ordered"), drift_path, TaskSpec(target="y"),
    split_on="day", val_fraction=0.2, holdout_fraction=0.2,
)
for name, path in [("work", ordered.data_path), ("val", ordered.val_path),
                   ("holdout", ordered.holdout_path)]:
    d = load_table(path)["day"]
    print(f"{name:8s} days {d.min():3d}-{d.max():3d}  mean y {load_table(path)['y'].mean():7.2f}")

The mean of the target is different in each slice, and no model chose that. A gap measured
across those rows is partly the search and partly the calendar.

The report says so in the run's own words whenever `split_on` is set, because a reader who
takes an ordered gap as pure selection optimism is reading it wrong. Rossmann v5 already
showed the sign can flip from this alone: its ordered exam came out **pessimistic** by
0.0026, where every random-split run before it had been optimistic.

In [ ]:
from tabular_agent.report import build_report

demo = build_report(
    task_type="regression", metric="rmse", selection_score=0.5, cv_score=0.53,
    fold_metrics=[0.5, 0.52], baseline_score=3.0, model_types=["lgbm"],
    feature_names=["x"], split=ordered.split_record(),
    holdout={"n_rows": 180, "scores": [
        {"experiment_id": "e1", "model_type": "lgbm", "selected": True,
         "val_score": 0.5, "holdout_score": 0.62, "optimism": -0.12},
    ]},
)
print(demo[demo.index("> **The gap is not purely selection"):])

## 9. What this does not fix

Worth being blunt, because a new number invites more confidence than it earns:

- **The search is not better, only the report is.** The agent still ranks tables on the
  validation half and still chases noise there. The sealed split tells the *reader
  afterwards* how much that cost. `0022` is what addresses the search itself.
- **Rows are not free.** At `holdout_fraction=0.2` house-prices' work half falls 1168 → 876.
  The agent builds on 25% less data, and 0033 already found a linear model at ~930 rows is on
  a steep learning curve. That is why this is opt-in per run rather than a default.
- **The seal is an allowlist, not physics.** The labels are on the same machine, in the same
  run directory, in a file the agent could name if `resolve_table` ever regressed.
  `wandb_fc/tabular-nomad` keeps its held-out labels on a *different machine*. That is the
  stronger version and we have not built it.

## 10. And only now, the LLM

Nothing about the agent's side changes. Same tools, same prompt, same menu — the third split
is entirely the harness's business (0029), which is why it takes one constructor argument
and no new instruction.

The cell below is the whole difference. It needs a gateway key; leave it unrun if you are
just reading.

In [ ]:
# from tabular_agent.harness.agent import build_agent, Deps, gateway_model
#
# run = RunSession(
#     os.path.join(workdir, "agent_run"), data_path, TaskSpec(target="price"),
#     holdout_fraction=0.2,          # <- the entire change
# )
# agent = build_agent(gateway_model("fireworks/deepseek-v4-flash-0731"))
# result = agent.run_sync("Build the best model you can.", deps=Deps(session=run))
#
# journal.load(run.run_dir)["holdout"]   # scored once, after the agent stopped
print("the agent's menu is unchanged:")
from tabular_agent.harness.agent import build_agent
print(sorted(build_agent()._function_toolset.tools))